# AI-Powered Aba Craft Product Intelligence System
## Model Training Notebook

This notebook demonstrates the training pipeline for both:
1. **Computer Vision Model**: Classifying Aba craft products (Bags, Shoes, Clothing, Accessories) using transfer learning with `EfficientNetB0`.
2. **NLP Model**: Sentiment analysis of customer reviews using `TF-IDF` and `Logistic Regression`.

---

### Step 1: Import Dependencies

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string

print("TensorFlow version:", tf.__version__)
print("NLTK version:", nltk.__version__)

### Step 2: Download NLTK Resources

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

### Step 3: Text Preprocessing Demonstration

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.word_tokenize(text)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(cleaned_tokens)

sample_review = "The handmade leather quality of this Aba-made bag is amazing!"
print("Original:", sample_review)
print("Preprocessed:", preprocess_text(sample_review))

### Step 4: NLP Sentiment Model Training

In [ ]:
# Load synthetic dataset (or real dataset once collected)
data_path = '../data/customer_reviews.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    # Fallback synthetic data creation
    reviews_data = [
        ("Excellent quality bag, leather feels premium.", 1),
        ("Love the design and size of the shoe.", 1),
        ("The dress fits perfectly and fabric is beautiful.", 1),
        ("Terrible shoe quality. It tore after one wear.", 0),
        ("The clothing shrank after first wash. Disappointed.", 0),
        ("The bag zipper is stuck and feels very cheap.", 0)
    ]
    df = pd.DataFrame(reviews_data, columns=["review", "sentiment"])

df['cleaned'] = df['review'].apply(preprocess_text)
print(df.head())

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['cleaned'])
y = df['sentiment'].values

sentiment_model = LogisticRegression()
sentiment_model.fit(X, y)
print("Sentiment analyzer model trained successfully!")

### Step 5: Computer Vision Transfer Learning Model (EfficientNetB0)

Here we show how transfer learning with `EfficientNetB0` is set up. For quick prototyping in this notebook, we initialize a model with random weights and train on dummy synthetic inputs.

In [ ]:
CLASSES = ["Bags", "Shoes", "Clothing", "Accessories"]
NUM_CLASSES = len(CLASSES)

# Create base model (EfficientNetB0)
base_model = applications.EfficientNetB0(weights=None, include_top=False, input_shape=(224, 224, 3))

# Add classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')


### Step 6: Compile and Save Model

In [ ]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Save training artifacts
os.makedirs('../models', exist_ok=True)
model.save('../models/product_classifier.keras')
print("Model saved to ../models/product_classifier.keras")